# Urdu translation + audiobook pipeline — open-weight edition

This notebook downloads open-weight models into the Colab runtime. No OpenAI API key is required and model weights are **not** stored in Git.

**Plan for a free Colab GPU**
- Translation: Qwen3-4B, loaded in 4-bit NF4 mode.
- Urdu speech: OmniVoice Urdu fine-tune (12,000 steps), a 0.6B TTS checkpoint.
- Models run in separate subprocesses so they do not need to occupy GPU memory at the same time.
- Sentence-level MP3s and a manifest keep transcript/audio aligned for sentence highlighting.

Model-card benchmarks are small self-reported tests, not independent quality guarantees. We audition a sample first; the actual runtime must be tested on your assigned Colab GPU before processing whole books.

In [ ]:
import os, sys, json, subprocess, pathlib, textwrap, gc
from google.colab import files
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable a GPU runtime")
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU before continuing."

REPO_URL = "https://github.com/mahashemi/BookReadingFramework.git"
PIPELINE_BRANCH = "feature/urdu-translation-pipeline"  # Use main after PR #130 is merged.
repo = pathlib.Path("/content/BookReadingFramework")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", PIPELINE_BRANCH, REPO_URL, str(repo)], check=True)

# Retain Colab's preinstalled PyTorch/CUDA build.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.51.0", "accelerate", "bitsandbytes",
                "sentencepiece", "soundfile", "omnivoice"], check=True)
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)

PIPELINE = repo / "tools/urdu_audiobook/local_pipeline.py"
print("Setup ready. Model weights download on first use.")

## 1. Upload one source chapter

Use a .txt or .md file you have permission to translate. For this first test, use a short representative passage from Book 02 (about 1–2 pages), not a whole book.

In [ ]:
uploaded = files.upload()
assert uploaded, "Upload a source .txt or .md file."
source_name = next(iter(uploaded))
source_path = pathlib.Path("/content") / source_name
assert source_path.suffix.lower() in {".txt", ".md"}, "Please upload a .txt or .md file."
output_dir = pathlib.Path("/content/urdu_audiobook_output")
output_dir.mkdir(exist_ok=True)
print("Source:", source_path, "| Output:", output_dir)

## 2. Translate locally with Qwen3-4B

The model is downloaded from Hugging Face and loaded in 4-bit NF4 mode to fit more comfortably on a free Colab GPU. The output preserves source/Urdu sentence pairs and flags every sentence for review. It does not call a paid inference API.

In [ ]:
subprocess.run([sys.executable, str(PIPELINE), "translate",
                "--input", str(source_path), "--output-dir", str(output_dir),
                "--batch-size", "3"], check=True)
translation_path = output_dir / "translation.json"
draft = json.loads(translation_path.read_text(encoding="utf-8"))
print("Aligned sentence pairs:", len(draft["sentences"]))
for item in draft["sentences"][:5]:
    print("\nSOURCE:", item["source_text"])
    print("URDU:  ", item["text_ur"])
files.download(str(translation_path))

## 3. Review the translation before generating speech

Download translation.json, review the Urdu with a fluent reader, then upload the corrected JSON in the next cell. Check especially religious terms, names/honorifics, Qur’an and hadith quotations, references, omissions, and natural Urdu. **Do not mark the file reviewed merely to bypass the gate.** Keep each source sentence beside its Urdu sentence; do not reorder or remove IDs.

In [ ]:
review_upload = files.upload()  # Upload reviewed translation.json after editing it.
assert "translation.json" in review_upload, "Upload the reviewed file with the exact name translation.json."
reviewed_path = pathlib.Path("/content/translation.json")
reviewed_path.write_bytes(review_upload["translation.json"])
reviewed = json.loads(reviewed_path.read_text(encoding="utf-8"))
assert len(reviewed.get("sentences", [])) == len(draft["sentences"]), "Sentence count changed; reconcile alignment."
assert [x["id"] for x in reviewed["sentences"]] == [x["id"] for x in draft["sentences"]], "Sentence IDs/order changed."
assert all(x.get("text_ur", "").strip() for x in reviewed["sentences"]), "An Urdu sentence is empty."
reviewed["review_status"] = "reviewed"
for item in reviewed["sentences"]:
    item["review_status"] = "reviewed"
translation_path.write_text(json.dumps(reviewed, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Reviewed translation accepted for this run. Continue to voice audition.")

## 4. Audition the Urdu voice

This creates a short local sample using OmniVoice Urdu. If you do not supply a reference recording, the exact generated preview becomes the voice reference for the chapter so all sentence segments use one consistent narrator. For a consistent narrator you already have permission to use, provide a **3–10 second Urdu recording** plus its exact transcript. Never clone a person's voice without consent.

In [ ]:
USE_REFERENCE_VOICE = False
PREVIEW_TEXT = "السلام علیکم۔ آج ہم اس کتاب کے ایک اہم موضوع کو سمجھیں گے۔"
REFERENCE_AUDIO_PATH = None
REFERENCE_TRANSCRIPT = None

if USE_REFERENCE_VOICE:
    ref_upload = files.upload()  # Upload a short authorized .wav/.mp3 recording.
    assert ref_upload, "Upload a permitted reference voice clip."
    REFERENCE_AUDIO_PATH = str(pathlib.Path("/content") / next(iter(ref_upload)))
    REFERENCE_TRANSCRIPT = input("Type the exact Urdu transcript spoken in the reference clip: ").strip()
    assert REFERENCE_TRANSCRIPT, "Reference transcript is required."

preview_path = pathlib.Path("/content/urdu_voice_preview.wav")
cmd = [sys.executable, str(PIPELINE), "preview", "--output", str(preview_path), "--text", PREVIEW_TEXT]
if USE_REFERENCE_VOICE:
    cmd += ["--reference-audio", REFERENCE_AUDIO_PATH, "--reference-text", REFERENCE_TRANSCRIPT]
subprocess.run(cmd, check=True)
from IPython.display import Audio, display
display(Audio(filename=str(preview_path)))
print("Listen for pronunciation, pacing, Urdu fluency, and religious terms before running the full chapter.")

## 5. Generate sentence-level audio locally

This downloads/loads the 0.6B Urdu OmniVoice checkpoint and generates one segment per reviewed Urdu sentence. It also creates audio_manifest.json, an RTL review.html, and a concatenated chapter.mp3 when FFmpeg succeeds. The first download may take several minutes; free Colab GPU availability and runtime limits vary.

In [ ]:
cmd = [sys.executable, str(PIPELINE), "audio",
       "--translation", str(translation_path), "--output-dir", str(output_dir)]
if USE_REFERENCE_VOICE:
    cmd += ["--reference-audio", REFERENCE_AUDIO_PATH, "--reference-text", REFERENCE_TRANSCRIPT]
else:
    # Reuse the auditioned synthetic sample as the speaker reference for consistency.
    cmd += ["--reference-audio", str(preview_path), "--reference-text", PREVIEW_TEXT]
subprocess.run(cmd, check=True)
print("Audio generation complete:", output_dir)

## 6. Package the results

Keep this first output as a pilot. Listen to the full sample and verify the spoken Urdu against the transcript. Whisper can optionally be added later as an ASR check, but ASR alone cannot verify religious meaning or guarantee correct pronunciation. Do not publish until editorial review is complete.

In [ ]:
subprocess.run([sys.executable, str(PIPELINE), "package", "--output-dir", str(output_dir)], check=True)
files.download(str(output_dir / "urdu_audiobook_bundle.zip"))